# ⚠️ DEPRECATED — no longer part of the pipeline

This notebook's job (regex-based field extraction) was merged into **Notebook 2** along with classification, and both now run via the Claude API instead of regex/keyword rules — this fixed real accuracy problems the regex approach had (documents using different label wording than expected, e.g. "FOR" instead of "Patient Name", "(Inscription)" instead of "Medications", never extracted correctly).

This notebook is **not run by the Databricks Job anymore** (removed from the task graph). Kept in the repo for reference/history only — do not re-add it to the job without updating it to match Notebook 2's current table schemas.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from dataclasses import dataclass, field, fields, asdict, MISSING
from typing import List, Optional
import re

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

catalog = "cdac-project"

# The 3 tables that exist BEFORE a document's type is known (raw
# file listing, extracted text, and the classification step itself)
# stay together in the original shared schema — they aren't, and
# can't be, organized by document type.
shared_schema = "intelligent-main-folder"
classification_table = f"`{catalog}`.`{shared_schema}`.document_classification"

# Output: one schema PER DOCUMENT TYPE, each with its own table,
# instead of everything piling into one shared schema. Each schema is
# created here if it doesn't exist yet — see OUTPUT_CONFIG in
# section 11, which ties each document_type to its table.
DOCUMENT_TYPE_SCHEMAS = {
    "RESUME": "resume",
    "EMAIL": "email",
    "INVOICE": "invoice",
    "BANK_STATEMENT": "bank_statement",
    "PRESCRIPTION": "prescription",
}

for schema_name in DOCUMENT_TYPE_SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`")

resume_output_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.resume_structured_data"
email_output_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.email_structured_data"
invoice_output_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.invoice_structured_data"
bank_statement_output_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.bank_statement_structured_data"
prescription_output_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.prescription_structured_data"

# Every document_type this notebook currently knows how to read from
# the classification table. This has to be a plain list rather than
# derived from EXTRACTOR_REGISTRY, because section 3 (read) runs
# before section 5 (where the registry is actually built) — when
# adding a new type, add it here too.
SUPPORTED_DOCUMENT_TYPES = ["RESUME", "EMAIL", "INVOICE", "BANK_STATEMENT", "PRESCRIPTION"]

In [ ]:
# ============================================================
# 3. READ INPUT TABLE
# ============================================================

classified_df = spark.table(classification_table).filter(
    F.col("document_type").isin(SUPPORTED_DOCUMENT_TYPES) &
    F.col("raw_text").isNotNull()
)

# Incremental — skip file_ids that already have a structured row in
# ANY output table, so re-running this notebook doesn't redo (and
# duplicate) work already done. file_id is a hash of the file's path
# + size (from Notebook 1), so it's unique regardless of document
# type — checking the union of every known output table is enough.
all_output_tables = (
    resume_output_table,
    email_output_table,
    invoice_output_table,
    bank_statement_output_table,
    prescription_output_table,
)

already_processed_ids = set()

for output_table in all_output_tables:
    if spark.catalog.tableExists(output_table):
        already_processed_ids |= {
            row.file_id
            for row in spark.table(output_table).select("file_id").collect()
        }

to_process_df = (
    classified_df.filter(~F.col("file_id").isin(already_processed_ids))
    if already_processed_ids
    else classified_df
)

display(to_process_df)

In [ ]:
# ============================================================
# 4. SCHEMA REGISTRY
# ============================================================
# A schema only DESCRIBES the fields a document type should have —
# it never performs extraction itself (see ResumeSchema in section 6
# for what a schema actually looks like). New document types are
# added by defining a dataclass like ResumeSchema and calling
# register_schema(...) at the bottom of it. Nothing here changes.

SCHEMA_REGISTRY = {}

def register_schema(document_type, schema_class):
    SCHEMA_REGISTRY[document_type] = schema_class

In [ ]:
# ============================================================
# 5. EXTRACTOR REGISTRY
# ============================================================
# This section has three parts:
#   - small reusable helpers (regex patterns, section splitting)
#   - BaseExtractor: the generic, schema-driven extraction engine
#   - the (currently empty) EXTRACTOR_REGISTRY itself
#
# None of this is document-type-specific — every extractor (Resume,
# Email, Invoice, Bank Statement, Prescription) reuses the exact same
# regex-field / section-field machinery, just with different schema
# field metadata.

# Shared sub-patterns, reused across several fields below.
_DATE_ALTERNATIVES = r"(\d{4}-\d{1,2}-\d{1,2}|[A-Za-z]+ \d{1,2},? \d{4}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})"
_AMOUNT_PATTERN = r"(\$?[\d,]+(?:\.\d{2})?)"

# ---- regex patterns, shared by any field with strategy="regex" ----
REGEX_PATTERNS = {
    "email": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),

    # Requires a phone-style label ("Phone:", "Mobile:", "Contact:",
    # "Tel:", "Cell:") immediately before the digits. The earlier
    # version searched for ANY digit-group-shaped substring anywhere
    # in the text with no label required — testing found it happily
    # matched things like "Employee ID: 4587236901" or "Reference:
    # 2024 0115 4487" as if they were a phone number. Requiring a
    # label is how every other field here already works (invoice
    # number, dates, amounts, patient name, ...) and resumes reliably
    # label their phone number, so this closes the false-positive gap
    # without meaningfully reducing what it actually catches.
    "phone": re.compile(r"\b(?:Phone|Mobile|Contact|Tel|Cell)(?:\s*(?:Number|No\.?))?\s*[:\-]?\s*(\+?[\d\s().-]{7,20})", re.IGNORECASE),

    "linkedin": re.compile(r"(?:https?://)?(?:www\.)?linkedin\.com/in/[A-Za-z0-9\-_%]+/?", re.IGNORECASE),
    "github": re.compile(r"(?:https?://)?(?:www\.)?github\.com/[A-Za-z0-9\-_]+/?", re.IGNORECASE),

    # Notebook 2 always writes emails as "From: ...\nTo: ...\n
    # Subject: ...\nDate: ...\n\n<body>" regardless of whether the
    # source was .eml or .msg, so these header patterns work on both.
    "email_from": re.compile(r"^From:\s*(.+)$", re.MULTILINE | re.IGNORECASE),
    "email_to": re.compile(r"^To:\s*(.+)$", re.MULTILINE | re.IGNORECASE),
    "email_subject": re.compile(r"^Subject:\s*(.+)$", re.MULTILINE | re.IGNORECASE),
    "email_date": re.compile(r"^Date:\s*(.+)$", re.MULTILINE | re.IGNORECASE),

    # Invoice. \b before "Total" specifically matters — without it,
    # "Subtotal: $100.00" also matches the bare "Total" alternative,
    # since "total" is a substring of "Subtotal". \b only asserts
    # between a word char and a non-word char, and there's no such
    # boundary between the "b" and "t" in "Subtotal", so \b correctly
    # rules that case out while still matching a standalone "Total".
    # Date patterns accept ISO (2024-01-15), "January 15, 2024", and
    # 01/15/2024 / 01-15-2024 — the first version only handled the
    # latter two, missing machine-generated ISO-format invoice dates
    # entirely. Amount patterns no longer require a decimal part —
    # "Total Due: $500" wasn't matching at all before, since the
    # pattern demanded ".XX" and a round-number invoice has none.
    "invoice_number": re.compile(r"Invoice\s*(?:Number|No\.?|#)\s*[:\-]?\s*(\S+)", re.IGNORECASE),
    "invoice_date": re.compile(r"Invoice\s*Date\s*[:\-]?\s*" + _DATE_ALTERNATIVES, re.IGNORECASE),
    "due_date": re.compile(r"Due\s*Date\s*[:\-]?\s*" + _DATE_ALTERNATIVES, re.IGNORECASE),
    "total_amount": re.compile(r"\b(?:Total\s*Due|Amount\s*Due|Balance\s*Due|Total)\s*[:\-]?\s*" + _AMOUNT_PATTERN, re.IGNORECASE),

    # Bank statement
    "bank_account_number": re.compile(r"Account\s*(?:Number|No\.?|#)\s*[:\-]?\s*(\S+)", re.IGNORECASE),
    "statement_period": re.compile(r"Statement\s*Period\s*[:\-]?\s*(.+)", re.IGNORECASE),
    "opening_balance": re.compile(r"Opening\s*Balance\s*[:\-]?\s*" + _AMOUNT_PATTERN, re.IGNORECASE),
    "closing_balance": re.compile(r"Closing\s*Balance\s*[:\-]?\s*" + _AMOUNT_PATTERN, re.IGNORECASE),

    # Prescription. \b before "Patient" avoids matching inside a word
    # like "outpatient" the same way the invoice pattern avoids
    # "Subtotal".
    "patient_name": re.compile(r"\bPatient(?:\s*Name)?\s*[:\-]\s*(.+)", re.IGNORECASE),
    "physician_name": re.compile(r"\b(?:Physician|Doctor|Prescribed\s*by)\s*[:\-]\s*(.+)", re.IGNORECASE),
    "prescription_date": re.compile(r"\bDate\s*[:\-]\s*" + _DATE_ALTERNATIVES, re.IGNORECASE),
    "diagnosis": re.compile(r"\bDiagnosis\s*[:\-]\s*(.+)", re.IGNORECASE),
}

# Used only to stop the name-heuristic below from mistaking a section
# header line for a person's name.
COMMON_SECTION_KEYWORDS = {
    "summary", "objective", "career objective", "about me",
    "skills", "technical skills", "skill set", "core competencies",
    "education", "academic background", "educational qualifications",
    "experience", "work experience", "professional experience", "employment history",
    "projects", "academic projects", "personal projects",
    "certifications", "certificates", "licenses",
}


def extract_name_heuristic(raw_text):
    """
    Best-effort guess at a person's name: the first short,
    letters-only line near the top of the document that isn't
    contact info or a section header. There's no reliable label for
    "name" the way there is for email/phone, so this is a heuristic —
    it returns None rather than a wrong guess when nothing plausible
    is found near the top.
    """

    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]

    for line in lines[:5]:
        if REGEX_PATTERNS["email"].search(line) or REGEX_PATTERNS["phone"].search(line):
            continue
        if line.strip(":").lower() in COMMON_SECTION_KEYWORDS:
            continue
        if len(line) > 60:
            continue

        words = line.split()
        if 1 <= len(words) <= 5 and all(w.strip(".,").isalpha() for w in words):
            return line

    return None


def extract_email_body(raw_text):
    """
    Everything after the first blank line in Notebook 2's email
    format is the body — the header block (From/To/Subject/Date)
    comes first, separated from the body by one blank line.
    """

    if not raw_text:
        return None

    parts = raw_text.split("\n\n", 1)
    return parts[1].strip() if len(parts) == 2 and parts[1].strip() else None


HEURISTIC_HANDLERS = {
    "extract_name": extract_name_heuristic,
    "extract_email_body": extract_email_body,
}


def split_into_sections(raw_text, section_field_specs):
    """
    Scans raw_text line by line for lines matching one of each
    field's known section-header aliases (e.g. "skills", "technical
    skills"), then slices the text between consecutive headers.

    Returns (sections, preamble):
      sections  - {field_name: block_text} for every header found
      preamble  - text before the first detected header (used as a
                  summary fallback when there's no explicit header)

    This is a heuristic, not real layout analysis — it expects each
    section header to sit on its own line, with nothing else on that
    line. That covers most real resumes/statements/prescriptions, but
    not a multi-column table header row (e.g. an invoice's
    "Description   Qty   Price   Amount" line) — which is exactly why
    InvoiceSchema (section 6C) doesn't use any section-strategy
    fields for line items; a heuristic that can't reliably find that
    boundary would silently over-capture instead.
    """

    alias_to_field = {}
    for field_name, aliases in section_field_specs:
        for alias in aliases:
            alias_to_field[alias.strip().lower()] = field_name

    lines = raw_text.splitlines()

    header_hits = []
    for i, line in enumerate(lines):
        cleaned = line.strip().rstrip(":").strip().lower()
        if cleaned in alias_to_field:
            header_hits.append((i, alias_to_field[cleaned]))

    preamble_end = header_hits[0][0] if header_hits else len(lines)
    preamble = "\n".join(lines[:preamble_end]).strip()

    sections = {}
    for idx, (line_no, field_name) in enumerate(header_hits):
        start = line_no + 1
        end = header_hits[idx + 1][0] if idx + 1 < len(header_hits) else len(lines)
        block = "\n".join(lines[start:end]).strip()

        if block and field_name not in sections:
            sections[field_name] = block

    return sections, preamble


def clean_preamble_for_summary(preamble, name):
    """
    Strips the name and any contact-info lines (email/phone/
    linkedin/github) out of the preamble before it's used as a
    summary fallback — otherwise a resume with no explicit "Summary"
    section ends up with the name and contact block repeated inside
    the summary text.
    """

    cleaned_lines = []

    for line in preamble.splitlines():
        stripped = line.strip()

        if not stripped:
            continue
        if name and stripped == name:
            continue
        if REGEX_PATTERNS["email"].search(stripped):
            continue
        if REGEX_PATTERNS["phone"].search(stripped):
            continue
        if REGEX_PATTERNS["linkedin"].search(stripped) or REGEX_PATTERNS["github"].search(stripped):
            continue

        cleaned_lines.append(stripped)

    return "\n".join(cleaned_lines).strip()


def format_section_value(raw_block, value_type):
    """Turns a raw section's text into the type its schema field expects."""

    if not raw_block:
        return [] if value_type in ("list_csv", "list_lines") else None

    if value_type == "list_csv":
        parts = re.split(r"[,;\n•▪●\-]+", raw_block)
        return [p.strip() for p in parts if p.strip()]

    if value_type == "list_lines":
        lines = []
        for raw_line in raw_block.splitlines():
            cleaned = raw_line.strip(" -•▪●\t").strip()
            # Some PDFs (icon-style bullet fonts) put the bullet glyph
            # on its own line, separate from the sentence that follows
            # it — that glyph often isn't one of the common bullet
            # characters stripped above, so it survives as a lone
            # "line" with no real content. Rather than enumerate every
            # possible bullet character, drop any line that has no
            # letters or digits at all — that's never real text.
            if cleaned and any(ch.isalnum() for ch in cleaned):
                lines.append(cleaned)
        return lines

    return raw_block.strip()  # value_type == "text"


class BaseExtractor:
    """
    Generic, schema-driven extraction engine. A concrete extractor
    for a document type only needs to set `schema_class` — this base
    class reads that schema's per-field metadata (strategy, regex
    pattern name, section aliases, ...) and knows how to populate
    every field without any field-by-field hardcoded logic. This is
    what makes adding a new document type require zero changes here:
    only a new schema (section 6-style) and a one-line extractor
    subclass (section 7-style).
    """

    schema_class = None

    def extract(self, raw_text):
        if self.schema_class is None:
            raise NotImplementedError("Extractor must set schema_class")

        raw_text = raw_text or ""

        schema_fields = fields(self.schema_class)

        section_specs = [
            (f.name, f.metadata["aliases"])
            for f in schema_fields
            if f.metadata.get("strategy") == "section"
        ]
        sections, preamble = split_into_sections(raw_text, section_specs)

        # Fields are processed in declaration order, so "name" (a
        # heuristic field declared first on ResumeSchema) is already
        # resolved by the time a later "section" field with
        # fallback_to_preamble needs it for cleanup. See
        # clean_preamble_for_summary.
        values = {}
        for f in schema_fields:
            strategy = f.metadata.get("strategy")

            if strategy == "regex":
                values[f.name] = self._extract_regex_field(raw_text, f)

            elif strategy == "section":
                block = sections.get(f.name, "")
                if not block and f.metadata.get("fallback_to_preamble"):
                    block = clean_preamble_for_summary(preamble, values.get("name"))
                values[f.name] = format_section_value(block, f.metadata.get("value_type"))

            elif strategy == "heuristic":
                values[f.name] = HEURISTIC_HANDLERS[f.metadata["handler"]](raw_text)

            else:
                values[f.name] = f.default_factory() if f.default_factory is not MISSING else f.default

        return self.schema_class(**values)

    def _extract_regex_field(self, raw_text, f):
        match = REGEX_PATTERNS[f.metadata["pattern"]].search(raw_text)

        if not match:
            return None

        # Use the first capture group if the pattern defines one
        # (e.g. email headers, where we want the value after "From:"
        # rather than "From:" itself) — otherwise fall back to the
        # whole match (resume patterns like email/phone have no
        # groups, so this preserves their original behavior exactly).
        value = match.group(1).strip() if match.lastindex else match.group(0).strip()

        normalize_name = f.metadata.get("normalize")
        if value and normalize_name:
            value = NORMALIZERS[normalize_name](value)

        validator_name = f.metadata.get("validator")
        if validator_name:
            value = VALIDATORS[validator_name](value)

        return value


EXTRACTOR_REGISTRY = {}

def register_extractor(document_type, extractor_instance):
    EXTRACTOR_REGISTRY[document_type] = extractor_instance

In [ ]:
# ============================================================
# 6. RESUME SCHEMA
# ============================================================
# Only describes the fields — no extraction logic lives here. Each
# field's metadata tells BaseExtractor HOW to fill it in:
#   strategy="regex"     -> look up REGEX_PATTERNS[pattern], validate/normalize
#   strategy="section"   -> find one of `aliases` as a section header
#   strategy="heuristic" -> call HEURISTIC_HANDLERS[handler]
#
# To add a field later (e.g. "languages" or "awards"), add one line
# here — ResumeExtractor (section 7) needs ZERO changes.

@dataclass
class ResumeSchema:
    name: Optional[str] = field(
        default=None,
        metadata={"strategy": "heuristic", "handler": "extract_name"}
    )
    email: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "email", "validator": "validate_email"}
    )
    phone: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "phone", "validator": "validate_phone"}
    )
    linkedin: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "linkedin", "normalize": "normalize_url", "validator": "validate_url_http"}
    )
    github: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "github", "normalize": "normalize_url", "validator": "validate_url_http"}
    )
    skills: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["skills", "technical skills", "skill set", "core competencies"], "value_type": "list_csv"}
    )
    education: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["education", "academic background", "educational qualifications"], "value_type": "list_lines"}
    )
    experience: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["experience", "work experience", "professional experience", "employment history"], "value_type": "list_lines"}
    )
    projects: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["projects", "academic projects", "personal projects"], "value_type": "list_lines"}
    )
    certifications: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["certifications", "certificates", "licenses"], "value_type": "list_lines"}
    )
    summary: Optional[str] = field(
        default=None,
        metadata={"strategy": "section", "aliases": ["summary", "professional summary", "objective", "career objective", "about me"], "value_type": "text", "fallback_to_preamble": True}
    )


register_schema("RESUME", ResumeSchema)

In [ ]:
# ============================================================
# 6B. EMAIL SCHEMA
# ============================================================
# Same idea as ResumeSchema — just fields, no logic. Notebook 2
# writes every email (.eml or .msg) as a "From:/To:/Subject:/Date:"
# header block followed by a blank line and the body, so these
# fields all use "regex" strategy (pattern names ending in
# email_from/to/subject/date, defined in section 5) except body,
# which uses the "extract_email_body" heuristic.

@dataclass
class EmailSchema:
    sender: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "email_from"}
    )
    recipient: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "email_to"}
    )
    subject: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "email_subject"}
    )
    sent_date: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "email_date"}
    )
    body: Optional[str] = field(
        default=None,
        metadata={"strategy": "heuristic", "handler": "extract_email_body"}
    )


register_schema("EMAIL", EmailSchema)

In [ ]:
# ============================================================
# 6C. INVOICE SCHEMA
# ============================================================
# All 4 fields use "regex" strategy — invoices reliably label these
# ("Invoice Number:", "Total Due:", etc.), so regex works well here.
# Deliberately NOT included: bill-to address and line items. Both
# would need "section" strategy, and testing showed that fails on a
# typical invoice — a line-item table's header row (e.g.
# "Description  Qty  Price  Amount") isn't recognized as a section
# boundary by split_into_sections (see its docstring in section 5),
# so a "bill_to" section would silently swallow the entire rest of
# the document, including the line items and totals. Shipping a
# field that quietly grabs too much is worse than not having it.

@dataclass
class InvoiceSchema:
    invoice_number: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "invoice_number"}
    )
    invoice_date: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "invoice_date"}
    )
    due_date: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "due_date"}
    )
    total_amount: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "total_amount"}
    )


register_schema("INVOICE", InvoiceSchema)

In [ ]:
# ============================================================
# 6D. BANK STATEMENT SCHEMA
# ============================================================
# Unlike invoices, a bank statement's "Transaction History" header
# is typically the only thing on its line (no multi-column header
# row problem), so "transactions" as a section field tested clean —
# each transaction line becomes one list entry.

@dataclass
class BankStatementSchema:
    account_number: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "bank_account_number"}
    )
    statement_period: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "statement_period"}
    )
    opening_balance: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "opening_balance"}
    )
    closing_balance: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "closing_balance"}
    )
    transactions: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["transaction history", "transactions", "account summary"], "value_type": "list_lines"}
    )


register_schema("BANK_STATEMENT", BankStatementSchema)

In [ ]:
# ============================================================
# 6E. PRESCRIPTION SCHEMA
# ============================================================
# "medications" tested clean the same way "transactions" did — a
# standalone "Medications:" header with nothing else on its line.

@dataclass
class PrescriptionSchema:
    patient_name: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "patient_name"}
    )
    physician_name: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "physician_name"}
    )
    prescription_date: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "prescription_date"}
    )
    diagnosis: Optional[str] = field(
        default=None,
        metadata={"strategy": "regex", "pattern": "diagnosis"}
    )
    medications: List[str] = field(
        default_factory=list,
        metadata={"strategy": "section", "aliases": ["medications", "medication", "medicine", "rx"], "value_type": "list_lines"}
    )


register_schema("PRESCRIPTION", PrescriptionSchema)

In [ ]:
# ============================================================
# 7. RESUME EXTRACTOR
# ============================================================
# Its ONLY responsibility is populating ResumeSchema. There is no
# field-by-field logic here on purpose — every field is driven
# entirely by the metadata declared on ResumeSchema (section 6),
# via the generic engine in BaseExtractor (section 5).

class ResumeExtractor(BaseExtractor):
    schema_class = ResumeSchema


register_extractor("RESUME", ResumeExtractor())

# Adding a future document type looks like this (not run — example):
#
# class InvoiceExtractor(BaseExtractor):
#     schema_class = InvoiceSchema
#
# register_extractor("INVOICE", InvoiceExtractor())
#
# No other cell in this notebook needs to change.

In [ ]:
# ============================================================
# 7B. EMAIL EXTRACTOR
# ============================================================
# Same pattern as ResumeExtractor — set schema_class, nothing else.

class EmailExtractor(BaseExtractor):
    schema_class = EmailSchema


register_extractor("EMAIL", EmailExtractor())

In [ ]:
# ============================================================
# 7C. INVOICE EXTRACTOR
# ============================================================

class InvoiceExtractor(BaseExtractor):
    schema_class = InvoiceSchema


register_extractor("INVOICE", InvoiceExtractor())

In [ ]:
# ============================================================
# 7D. BANK STATEMENT EXTRACTOR
# ============================================================

class BankStatementExtractor(BaseExtractor):
    schema_class = BankStatementSchema


register_extractor("BANK_STATEMENT", BankStatementExtractor())

In [ ]:
# ============================================================
# 7E. PRESCRIPTION EXTRACTOR
# ============================================================

class PrescriptionExtractor(BaseExtractor):
    schema_class = PrescriptionSchema


register_extractor("PRESCRIPTION", PrescriptionExtractor())

In [ ]:
# ============================================================
# 8. VALIDATION FUNCTIONS
# ============================================================
# Called by BaseExtractor for any field with a "validator" (or
# "normalize") entry in its metadata. A failed validation returns
# None instead of raising — see OBJECTIVE: "If validation fails,
# store NULL instead of crashing."

def validate_email(value):
    if value and re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", value):
        return value
    return None


def validate_phone(value):
    if not value:
        return None
    digits = re.sub(r"\D", "", value)
    return value if 7 <= len(digits) <= 15 else None


def validate_url_http(value):
    return value if value and value.lower().startswith("http") else None


def normalize_url(value):
    return value if value.lower().startswith("http") else f"https://{value}"


VALIDATORS = {
    "validate_email": validate_email,
    "validate_phone": validate_phone,
    "validate_url_http": validate_url_http,
}

NORMALIZERS = {
    "normalize_url": normalize_url,
}

In [ ]:
# ============================================================
# 9. PROCESSING PIPELINE
# ============================================================
# Runs the correct extractor (via EXTRACTOR_REGISTRY) for one
# document. This function is allowed to raise — per-record error
# handling lives separately, in section 10, so a bad document can't
# silently corrupt this function's logic.

def extract_structured_data(document_type, raw_text):
    extractor = EXTRACTOR_REGISTRY.get(document_type)

    if extractor is None:
        raise Exception(f"No extractor registered for document_type={document_type}")

    return extractor.extract(raw_text)

In [ ]:
# ============================================================
# 10. ERROR HANDLING
# ============================================================
# Processes one record end-to-end and always returns a dict — never
# raises. One failed document must not stop the batch, so any
# exception is captured into processing_status/error_message here
# instead of propagating up. A dict (not a fixed tuple) is used
# because different document types have different fields — RESUME
# and EMAIL rows don't share a shape, only file_id/document_type/
# processing_status/error_message are common to all of them.

# Default (empty) field values per document type, so a FAILED record
# still gets well-typed nulls/empty-lists instead of missing keys.
SCHEMA_FIELD_DEFAULTS = {
    document_type: {
        f.name: (f.default_factory() if f.default_factory is not MISSING else f.default)
        for f in fields(schema_class)
    }
    for document_type, schema_class in SCHEMA_REGISTRY.items()
}


def build_output_row(row):
    file_id = row["file_id"]
    document_type = row["document_type"]
    raw_text = row["raw_text"]

    try:
        structured = extract_structured_data(document_type, raw_text)
        values = asdict(structured)
        processing_status = "SUCCESS"
        error_message = None

    except Exception as e:
        values = dict(SCHEMA_FIELD_DEFAULTS.get(document_type, {}))
        processing_status = "FAILED"
        error_message = str(e)

    return {
        "file_id": file_id,
        "document_type": document_type,
        **values,
        "processing_status": processing_status,
        "error_message": error_message,
    }


# One collect() to bring the (already-filtered, already-incremental)
# rows to the driver for per-record Python extraction — the same
# pattern Notebooks 2 and 3 use, and intentionally the only collect()
# in this notebook.
results = [build_output_row(row) for row in to_process_df.collect()]

In [ ]:
# ============================================================
# 11. CREATE OUTPUT DATAFRAME
# ============================================================

RESUME_STRUCTURED_SCHEMA = """
file_id STRING,
document_type STRING,
name STRING,
email STRING,
phone STRING,
linkedin STRING,
github STRING,
skills ARRAY<STRING>,
education ARRAY<STRING>,
experience ARRAY<STRING>,
projects ARRAY<STRING>,
certifications ARRAY<STRING>,
summary STRING,
processing_status STRING,
error_message STRING
"""

EMAIL_STRUCTURED_SCHEMA = """
file_id STRING,
document_type STRING,
sender STRING,
recipient STRING,
subject STRING,
sent_date STRING,
body STRING,
processing_status STRING,
error_message STRING
"""

INVOICE_STRUCTURED_SCHEMA = """
file_id STRING,
document_type STRING,
invoice_number STRING,
invoice_date STRING,
due_date STRING,
total_amount STRING,
processing_status STRING,
error_message STRING
"""

BANK_STATEMENT_STRUCTURED_SCHEMA = """
file_id STRING,
document_type STRING,
account_number STRING,
statement_period STRING,
opening_balance STRING,
closing_balance STRING,
transactions ARRAY<STRING>,
processing_status STRING,
error_message STRING
"""

PRESCRIPTION_STRUCTURED_SCHEMA = """
file_id STRING,
document_type STRING,
patient_name STRING,
physician_name STRING,
prescription_date STRING,
diagnosis STRING,
medications ARRAY<STRING>,
processing_status STRING,
error_message STRING
"""

# One entry per document type this notebook writes output for.
# Adding a new type means adding one entry here (its own DDL schema
# + output table, alongside its schema/extractor in sections 6/7) —
# the loop below stays unchanged.
OUTPUT_CONFIG = {
    "RESUME": {
        "table": resume_output_table,
        "ddl_schema": RESUME_STRUCTURED_SCHEMA,
        "fields": ["name", "email", "phone", "linkedin", "github", "skills", "education", "experience", "projects", "certifications", "summary"],
    },
    "EMAIL": {
        "table": email_output_table,
        "ddl_schema": EMAIL_STRUCTURED_SCHEMA,
        "fields": ["sender", "recipient", "subject", "sent_date", "body"],
    },
    "INVOICE": {
        "table": invoice_output_table,
        "ddl_schema": INVOICE_STRUCTURED_SCHEMA,
        "fields": ["invoice_number", "invoice_date", "due_date", "total_amount"],
    },
    "BANK_STATEMENT": {
        "table": bank_statement_output_table,
        "ddl_schema": BANK_STATEMENT_STRUCTURED_SCHEMA,
        "fields": ["account_number", "statement_period", "opening_balance", "closing_balance", "transactions"],
    },
    "PRESCRIPTION": {
        "table": prescription_output_table,
        "ddl_schema": PRESCRIPTION_STRUCTURED_SCHEMA,
        "fields": ["patient_name", "physician_name", "prescription_date", "diagnosis", "medications"],
    },
}

# Group the generic per-record dicts (section 10) by document_type,
# since each type gets its own DataFrame/schema/table.
results_by_type = {}
for row_dict in results:
    results_by_type.setdefault(row_dict["document_type"], []).append(row_dict)

output_dataframes = {}

for document_type, rows in results_by_type.items():
    config = OUTPUT_CONFIG.get(document_type)

    if config is None:
        continue  # no output table configured for this type yet

    row_tuples = [
        (
            r["file_id"],
            r["document_type"],
            *[r.get(field_name) for field_name in config["fields"]],
            r["processing_status"],
            r["error_message"],
        )
        for r in rows
    ]

    df = (
        spark.createDataFrame(row_tuples, schema=config["ddl_schema"])
        .withColumn("processing_timestamp", F.current_timestamp())
    )

    output_dataframes[document_type] = df
    display(df)

In [ ]:
# ============================================================
# 12. WRITE DELTA TABLE
# ============================================================
# FIXED: was mode("overwrite"), which replaced each output table
# with ONLY the current run's batch every time — since section 3's
# incremental read already excludes file_ids from ANY existing
# output table, that meant every re-run silently DISCARDED every
# resume/email/invoice/etc. processed by earlier runs, keeping only
# whatever was new this time. Now mode("append"): section 3's
# incremental filter guarantees this run's rows never duplicate an
# existing file_id, so append is safe and actually accumulates
# results across runs instead of losing them.

def write_structured_data(document_type, df):
    table_name = OUTPUT_CONFIG[document_type]["table"]

    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)


for document_type, df in output_dataframes.items():
    write_structured_data(document_type, df)

In [ ]:
# ============================================================
# 13. DISPLAY FINAL OUTPUT
# ============================================================

for document_type, config in OUTPUT_CONFIG.items():
    if spark.catalog.tableExists(config["table"]):
        print(f"--- {document_type} ---")
        display(spark.table(config["table"]))